In [1]:
## Hybrid Retriever  - Combining Dense and Sparse Retriever

In [3]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

In [4]:
## Step 1: Sample Documents
docs = [
    Document(page_content="Langchain helps build LLM applications."),
    Document(page_content="Pinecone is a vector database for semantic search."),
    Document(page_content="The Eiffel Tower is located in Paris."),
    Document(page_content="Langchain can be used to develop agentic ai application."),
    Document(page_content="Langchain has many types of retrievers"),
] 

## Step 2: Dense Retriever (FAISS + HuggingFace)
embedding_model = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")
dense_vectorstore = FAISS.from_documents(docs, embedding_model)
dense_retriever = dense_vectorstore.as_retriever()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8232.12it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
## Sparse Retriever (BM25)
sparse_retriever  = BM25Retriever.from_documents(docs)
sparse_retriever.k=3  ##top -k  documents to retriever


## Step 4 : Combine wih Ensemble Retriever
hybrid_retriever  = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[0.7, 0.3],
)

In [7]:
hybrid_retriever

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000025601B30BF0>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x0000025627BD8470>, k=3)], weights=[0.5, 0.5])

In [8]:
## Step 5: Query and get results
query = "How can I build an application using LLMs?"
results = hybrid_retriever.invoke(query)

## Step 6: Print Results
for i, doc in enumerate(results):
    print(f"\n Document {i +1} : \n{doc.page_content}")


 Document 1 : 
Langchain helps build LLM applications.

 Document 2 : 
Langchain can be used to develop agentic ai application.

 Document 3 : 
Langchain has many types of retrievers

 Document 4 : 
Pinecone is a vector database for semantic search.


In [9]:
## RAG Pipeline with hybrid Retriever

In [ ]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

In [ ]:
## Step 5: Prompt Template
prompt  = PromptTemplate.from_template("""
Answer the question based on the context provided.
Context:
{context}
Question:
{input}
""")

## Step 6: LLM (requires OPENAI_API_KEY)
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.2)
llm


In [ ]:
## 